# [PBT] 화장품 이커머스 ㅣ 11. 다변량 · 변수 선택 〈둘이 함께〉

## 이 노트북이 하는 일

EDA **STEP 4 다변량 + STEP 5 변수 선택** — ④ 탐색적 데이터 분석의 마지막 장이다.
> 묻는 것 : **변수끼리 겹치는가**(채택 근거 ③) → **그래서 무엇을 모델에 넣는가**

```
09a·09b  단변량   →  변수 자체의 생김새
10a·10b  이변량   →  Y를 설명하는가        (채택 근거 ①·②)
                    ─────────────────────
                    11  다변량 → 판정 → 최종 X 집합      ← 이 노트북
```

`10a`(이슬기)와 `10b`(배지환)를 **나눠 본 두 사람이 여기서 처음 같은 표를 본다.**
나눌 수 없는 구간이다 — 변수 하나를 빼면 나머지 전부의 VIF가 다시 계산되기 때문이다(설계 §7 · §8 구간 3).

```
수치형 5 : price · prior_view_cnt · prior_uniq_product_cnt · cart_product_view_cnt · has_prior_view
명목형 3 : brand_grp · cart_timeband · cart_weekday   ← 공통 눈금만. VIF 는 ⑤ 로 넘긴다
```

## 여기서 하지 않는 것

| 안 하는 것 | 어디서 |
|---|---|
| 로그변환 · 구간화 · 인코딩 **실행** | ⑤ 전처리 — **층화 분할 후** (설계 §3-6 · LAB-13 04) |
| 명목형 더미 전개 후 VIF | ⑤ 모델링 (설계 §3-9) |
| 모델 적합 · 성능 평가 | ⑥ 모델 구축 |
| **Y 와의 상관분석** | **하지 않는다.** `purchase_7d`가 0/1이라 피어슨이 의미를 갖지 못한다 (설계 §2 ⛔) |

## 진입 전 확정된 것 — 결과를 보기 전에 정해져 있다

| 규칙 | 내용 | 출처 |
|---|---|---|
| VIF 임계값 | **10.0** | 설계 §3-9 |
| 자동 제거 금지 | `my_prep.reduce_vif`를 **쓰지 않는다.** 값만 보고 **판단은 사람이 한다** | 설계 §3-9 |
| 공통 눈금 | `Y 비율 최대 − 최소 (%p)` · **n ≥ 1,000 구간만** | 설계 §3-3 |
| 구간 경계 | 아래 표. **결과를 보고 바꾸지 않는다** | 설계 §3-3 |
| 단조성 | 중간 역전 폭 **1.0%p 이하**면 노이즈로 보아 단조로 간주 | 설계 §3-5 |
| 포함관계 | `prior_view_cnt` vs `has_prior_view`는 **별도 상대 규칙** | 설계 §3-4 |
| 로그변환 | `\|왜도\| >= 1.0` · 0을 포함하면 `log1p`, 아니면 `log` | 설계 §3-6 |

### 구간 경계 — 미리 고정한 것

| 변수 | 구간 |
|---|---|
| `prior_view_cnt` · `prior_uniq_product_cnt` | `0 / 1 / 2 / 3 / 4-5 / 6-10 / 11+` |
| `cart_product_view_cnt` | `0 / 1 / 2 / 3 / 4+` |
| `price` | **5분위** |
| `has_prior_view` · 명목형 3종 | 범주 그대로 |

> **구간을 잘게 쪼갤수록 최대−최소가 자동으로 커진다.** 결과를 보고 경계를 조정하면 눈금 자체가 무의미해지므로
> 경계는 07번·설계 §3-3에서 이미 확정된 것을 그대로 쓴다.

## 판정 규율 (설계 §3-8)

**효과크기로 자르지 않는다.** 자르는 근거는 **셋뿐**이다.

| 칸 | 조건 |
|---|---|
| **제외** | ① 정보 누수 ② **다변량에서 중복으로 판명** ③ 설계 단계에서 이미 제외 결정 |
| **보류** | 이변량만으로 결정할 수 없다 → 다변량 후 재판정 |
| **채택** | 누수 없음 + 중복 없음 + 계획서가 근거를 댄 변수 |

**시작 배치**

```
제외  lag_hours · category_grp · brand/category_id 원본     ← 이미 정해진 것
보류  prior_view_cnt · prior_uniq_product_cnt
      cart_product_view_cnt · has_prior_view               ← 오늘 판단할 것
채택  price · brand_grp · cart_timeband · cart_weekday
```

### 왜 효과가 작아도 자르지 않는가

1. X가 8개뿐이다. 표본 10만에 변수 8개면 **과적합 압력이 없다**
2. 계획 보고서 §3에 변수마다 *"왜 넣는가"*를 이미 써서 제출했다. 결과를 보고 빼면 **그 근거를 스스로 부정**하는 것이다
3. 결과 보고서 사양 §5 — *"설명이 목적이면 변수 추가 기준은 **해석이 되는가**"*. 효과 강도가 아니다
4. `brand_grp`은 `price`와의 **교란을 통제**하려고 넣은 변수다. 효과가 작아도 빠지면 안 된다

> **그러므로 오늘 실제로 자를 수 있는 근거는 ② 다변량 중복 하나뿐이다.**

### 중복일 때 누구를 남기는가 — 미리 정한 두 규칙

| 상황 | 규칙 | 출처 |
|---|---|---|
| VIF > 10 인 쌍 | **효과 강도가 더 높은 쪽을 남긴다.** 그 강도의 자는 **공통 눈금**(설계 §3-3) | LAB-13 3-7 · 설계 §3-9 |
| `prior_view_cnt` ↔ `has_prior_view` | **포함관계 별도 규칙** — `A > B`면 건수, `A <= B`면 이진 | 설계 §3-4 |

## ⚠️ 공통 눈금의 한계 — 보고서에도 적는다

구간 개수에 영향을 받는다. `brand_grp`(15종)는 `cart_timeband`(4구간)보다 **구조적으로 유리하다.**
**성격이 비슷한 변수끼리의 상대 비교에만 쓰고, 절대 크기 판정에는 쓰지 않는다**(설계 §3-3).

## 📐 수치 표기 규칙 (설계 §4-5)

> 판정문의 모든 수치에 **어느 셀에서 나오는지**를 `#03-1` 형태로 적는다.
> 앞 노트북의 CSV에서 온 것은 `10a` 처럼 파일을 밝힌다. **둘 다 아닌 수치는 쓰지 않는다.**

---

## #01. 준비

In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)
sys.path.insert(0, BASE)

from numpy import where, inf
from pandas import (read_csv, to_datetime, DataFrame, Series, cut, qcut,
                    set_option, concat)
from helpers import my_stats, my_prep, my_plot

set_option("display.max_columns", 50)
set_option("display.max_colwidth", 70)

OUT = BASE + r"\output"

ALPHA = 0.05
VIF_MAX = 10.0          # 설계 §3-9
MIN_N = 1000            # 설계 §3-3 — 공통 눈금에 쓸 구간의 최소 인원
NOISE = 1.0             # 설계 §3-5 — 단조성 노이즈 허용폭 (%p)
SKEW_MAX = 1.0          # 설계 §3-6 — 로그변환 대상 왜도
print("데이터 폴더 :", OUT)

데이터 폴더 : .\output


### 1-1. 마스터 테이블 + 08번 대조 — 하나라도 어긋나면 멈춘다

In [2]:
m = read_csv(OUT + r"\03_master.csv", encoding="utf-8-sig")

m["t0"]          = to_datetime(m["t0"], format="%Y-%m-%d %H:%M:%S")
m["product_id"]  = m["product_id"].astype("int32")
m["purchase_7d"] = m["purchase_7d"].astype("int8")
m["cart_timeband"] = m["cart_timeband"].astype("category").cat.set_categories(
    ["심야", "오전", "오후", "저녁"], ordered=True)
m["cart_weekday"]  = m["cart_weekday"].astype("category").cat.set_categories(
    ["월", "화", "수", "목", "금", "토", "일"], ordered=True)
for c in ["brand_grp", "category_grp"]:
    m[c] = m[c].astype("category")

BASE_RATE = m["purchase_7d"].mean() * 100

NUM4 = ["price", "prior_view_cnt", "prior_uniq_product_cnt", "cart_product_view_cnt"]
NUM5 = NUM4 + ["has_prior_view"]
NOMI3 = ["brand_grp", "cart_timeband", "cart_weekday"]
X8 = NUM5 + NOMI3

ok = [("행 수", f"{len(m):,}", "108,458"),
      ("1행 = 1고객", str(len(m) == m["user_id"].nunique()), "True"),
      ("Y 양성", f"{int(m['purchase_7d'].sum()):,}명 / {m['purchase_7d'].mean():.2%}", "22,236명 / 20.50%"),
      ("brand_grp", f"{m['brand_grp'].nunique()}종", "15종")]
chk = DataFrame(ok, columns=["항목", "이번 실행", "08번 값"])
chk["일치"] = chk["이번 실행"] == chk["08번 값"]
chk

,항목,이번 실행,08번 값,일치
0,행 수,"108,458","108,458",True
1,1행 = 1고객,True,True,True
2,Y 양성,"22,236명 / 20.50%","22,236명 / 20.50%",True
3,brand_grp,15종,15종,True


### 1-2. 앞 노트북 결과 4개 수령 — 하나라도 없으면 여기서 멈춘다

In [3]:
U_A = read_csv(OUT + r"\09a_단변량_A조.csv", encoding="utf-8-sig", index_col=0)
U_B = read_csv(OUT + r"\09b_단변량_B조.csv", encoding="utf-8-sig", index_col="변수")
A_BIV = read_csv(OUT + r"\10a_이변량_A조.csv", encoding="utf-8-sig", index_col="변수")
B_BIV = read_csv(OUT + r"\10b_이변량_B조.csv", encoding="utf-8-sig", index_col="변수")
BIV = concat([A_BIV, B_BIV])

src = DataFrame([("09a 단변량 A조", len(U_A), 4), ("09b 단변량 B조", len(U_B), 5),
                 ("10a 이변량 A조", len(A_BIV), 4), ("10b 이변량 B조", len(B_BIV), 4)],
                columns=["파일", "행 수", "기대"])
src["일치"] = src["행 수"] == src["기대"]

print(f"이변량 {len(BIV)}종 == 설계 §1 'X는 8개' : "
      f"{'일치 ✅' if len(BIV) == 8 else '불일치 ❌'}")
print(f"10a A조 : {list(A_BIV.index)}")
print(f"10b B조 : {list(B_BIV.index)}")
src

이변량 8종 == 설계 §1 'X는 8개' : 일치 ✅
10a A조 : ['prior_view_cnt', 'prior_uniq_product_cnt', 'cart_product_view_cnt', 'has_prior_view']
10b B조 : ['price', 'brand_grp', 'cart_timeband', 'cart_weekday']


,파일,행 수,기대,일치
0,09a 단변량 A조,4,4,True
1,09b 단변량 B조,5,5,True
2,10a 이변량 A조,4,4,True
3,10b 이변량 B조,4,4,True


### 1-3. 09번 왜도와 대조 — 로그변환 판정의 근거를 여기서 확인한다

`09b`는 `price`에 평균·중앙값이 없고 `09a`는 컬럼명이 다르다(`skew` vs `왜도`).
**단변량 요약은 마스터에서 다시 계산하고, 왜도는 09번 값과 대조**해 어긋나지 않는지 확인한다.

In [4]:
skew_09 = {}
for c in NUM4:
    if c in U_A.index and "skew" in U_A.columns:
        skew_09[c] = float(U_A.loc[c, "skew"])
    elif c in U_B.index and "왜도" in U_B.columns:
        skew_09[c] = float(U_B.loc[c, "왜도"])

cmp = DataFrame([{"변수": c, "09번 왜도": round(skew_09[c], 2),
                  "여기서 재계산": round(float(m[c].skew()), 2)} for c in NUM4]).set_index("변수")
cmp["차이"] = (cmp["09번 왜도"] - cmp["여기서 재계산"]).abs().round(2)
cmp["일치(0.05 이내)"] = cmp["차이"] <= 0.05
cmp

,09번 왜도,여기서 재계산,차이,일치(0.05 이내)
변수,,,,
price,7.21,7.21,0.0,True
prior_view_cnt,5.90,5.90,0.0,True
prior_uniq_product_cnt,6.04,6.04,0.0,True
cart_product_view_cnt,1.75,1.75,0.0,True


---

## #02. 수치형 5종 상관행렬 — 겹치는 쌍을 찾는다

| 무엇을 | 왜 |
|---|---|
| 상관행렬 | 어느 **쌍**이 겹치는지 본다. VIF는 "겹친다"까지만 알려주고 **상대를 지목하지 못한다** |
| `plot=False` | `pairplot`은 108,458행이라 매우 느리다 (설계 §5 함정) |

> **`correlation`은 `abs(skew) > 1`이면 Spearman으로 자동 전환된다**(설계 §5 함정).
> 조회 변수는 왜도가 크므로 **대부분 Spearman이 나올 것이다.** 어느 계수인지 `method` 열로 확인한다.
>
> ⚠️ **Spearman(순위)과 VIF(선형회귀)는 서로 다른 자다.** 두 결과가 같은 방향을 가리키면
> 근거가 겹쳐 든든하지만, 어긋나면 **어느 쪽이 맞다고 단정하지 않고 둘 다 적는다.**

In [5]:
CORR = my_stats.multi_correlation(m, columns=NUM5, plot=False)
CORR[["method", "coef", "p-value", "strength", "significant", "high_skew"]]

,price,prior_view_cnt,prior_uniq_product_cnt,cart_product_view_cnt,has_prior_view
price,1.000,0.055,0.049,0.121,0.058
prior_view_cnt,0.055,1.000,0.988,0.688,0.918
prior_uniq_product_cnt,0.049,0.988,1.000,0.666,0.923
cart_product_view_cnt,0.121,0.688,0.666,1.000,0.703
has_prior_view,0.058,0.918,0.923,0.703,1.000


method    coef  p-value  \
x                      y                                                   
price                  prior_view_cnt          Spearman  0.0552      0.0   
                       prior_uniq_product_cnt  Spearman  0.0493      0.0   
                       cart_product_view_cnt   Spearman  0.1213      0.0   
                       has_prior_view          Spearman  0.0577      0.0   
prior_view_cnt         prior_uniq_product_cnt  Spearman  0.9881      0.0   
                       cart_product_view_cnt   Spearman  0.6877      0.0   
                       has_prior_view          Spearman  0.9177      0.0   
prior_uniq_product_cnt cart_product_view_cnt   Spearman  0.6664      0.0   
                       has_prior_view          Spearman  0.9231      0.0   
cart_product_view_cnt  has_prior_view          Spearman  0.7034      0.0   

                                               strength  significant  \
x                      y                                               
price                  prior_view_cnt              Weak         True   
                       prior_uniq_product_cnt      Weak         True   
                       cart_product_view_cnt       Weak         True   
                       has_prior_view              Weak         True   
prior_view_cnt         prior_uniq_product_cnt    Strong         True   
                       cart_product_view_cnt   Moderate         True   
                       has_prior_view            Strong         True   
prior_uniq_product_cnt cart_product_view_cnt   Moderate         True   
                       has_prior_view            Strong         True   
cart_product_view_cnt  has_prior_view            Strong         True   

                                               high_skew  
x                      y                                  
price                  prior_view_cnt               True  
                       prior_uniq_product_cnt       True  
                       cart_product_view_cnt        True  
                       has_prior_view               True  
prior_view_cnt         prior_uniq_product_cnt       True  
                       cart_product_view_cnt        True  
                       has_prior_view               True  
prior_uniq_product_cnt cart_product_view_cnt        True  
                       has_prior_view               True  
cart_product_view_cnt  has_prior_view               True

### 2-1. 중간 이상 상관만 추려 본다

`correlation_summary`의 기본값은 `"Strong"`(> 0.7)이라 중간 상관을 놓친다.
**`["Strong", "Moderate"]`로 넓혀 본다**(설계 §5 함정).

In [6]:
SUM_CORR = my_stats.correlation_summary(CORR, strength=["Strong", "Moderate"])
print("변수별 중복 신호  (count = |coef| > 0.3 인 상대의 수)")
display(SUM_CORR)

MID = CORR[CORR["strength"].isin(["Strong", "Moderate"])]
print(f"\n중간 이상(|coef| > 0.3) 상관 쌍 : {len(MID)}개")
display(MID[["method", "coef", "strength"]] if len(MID) else "해당 쌍 없음")

변수별 중복 신호  (count = |coef| > 0.3 인 상대의 수)


,max-y,max-coef,count,columns
x,,,,
prior_uniq_product_cnt,prior_view_cnt,0.9881,3,"prior_view_cnt, has_prior_view, cart_product_view_cnt"
prior_view_cnt,prior_uniq_product_cnt,0.9881,3,"prior_uniq_product_cnt, has_prior_view, cart_product_view_cnt"
has_prior_view,prior_uniq_product_cnt,0.9231,3,"prior_uniq_product_cnt, prior_view_cnt, cart_product_view_cnt"
cart_product_view_cnt,has_prior_view,0.7034,3,"has_prior_view, prior_view_cnt, prior_uniq_product_cnt"
price,cart_product_view_cnt,0.1213,0,-



중간 이상(|coef| > 0.3) 상관 쌍 : 6개


method    coef  strength
x                      y                                                 
prior_view_cnt         prior_uniq_product_cnt  Spearman  0.9881    Strong
                       cart_product_view_cnt   Spearman  0.6877  Moderate
                       has_prior_view          Spearman  0.9177    Strong
prior_uniq_product_cnt cart_product_view_cnt   Spearman  0.6664  Moderate
                       has_prior_view          Spearman  0.9231    Strong
cart_product_view_cnt  has_prior_view          Spearman  0.7034    Strong

---

## #03. VIF — 임계값 10.0 (설계 §3-9)

상관행렬은 **두 변수씩**만 본다. 셋 이상이 얽혀 만드는 중복은 VIF이어야 잡힌다.

> **`compute_vif`는 수치형만 받는다.** `category` dtype이 섞이면 `TypeError`가 난다(설계 §5 함정).
> 명목형 3종은 더미 전개가 필요하고, *"더미 인코딩·다중공선성은 **분할 후에 확인**"*해야 하므로,
> **⑤ 모델링 단계로 넘긴다**(설계 §3-9).

In [7]:
VIF = my_stats.compute_vif(m, columns=NUM5)
VIF["임계 초과"] = where(VIF["VIF"] > VIF_MAX, "★ 초과", "")
VIF.round(3)

,VIF,임계 초과
prior_view_cnt,20.218,★ 초과
prior_uniq_product_cnt,19.995,★ 초과
has_prior_view,2.051,
cart_product_view_cnt,2.031,
price,1.011,


### 3-1. 한 변수를 빼면 나머지가 어떻게 변하는가

VIF는 **X 집합 전체에서 나오는 값**이라, 변수 하나를 빼면 나머지 전부가 다시 계산된다.
그래서 *"이 변수를 빼면 해결되는가"*는 실제로 빼 보고 확인해야 한다.

> **이건 판정이 아니라 관찰이다.** 무엇을 뺄지는 `#07`에서 규칙으로 정한다.

In [8]:
rows = []
for drop in NUM5:
    keep = [c for c in NUM5 if c != drop]
    v = my_stats.compute_vif(m, columns=keep)
    rows.append({"뺀 변수": drop, "남은 변수 수": len(keep),
                 "최대 VIF": round(float(v["VIF"].max()), 3),
                 "최대인 변수": v["VIF"].idxmax(),
                 "임계 초과 수": int((v["VIF"] > VIF_MAX).sum())})

VIF_DROP = DataFrame(rows).set_index("뺀 변수")
print(f"전체 5개일 때 최대 VIF : {VIF['VIF'].max():.3f}  (임계 {VIF_MAX})")
VIF_DROP

전체 5개일 때 최대 VIF : 20.218  (임계 10.0)


,남은 변수 수,최대 VIF,최대인 변수,임계 초과 수
뺀 변수,,,,
price,4,20.213,prior_view_cnt,2
prior_view_cnt,4,1.922,has_prior_view,0
prior_uniq_product_cnt,4,1.817,has_prior_view,0
cart_product_view_cnt,4,16.927,prior_uniq_product_cnt,2
has_prior_view,4,18.946,prior_view_cnt,2


---

## #04. 공통 눈금 — 8개를 한 줄로 세우는 유일한 자 (설계 §3-3)

```
Y 비율 최대 − 최소 (%p)      단,  n >= 1,000 인 구간·범주만 사용
```

`np2`와 `V`는 척도가 달라 서로 비교할 수 없고, 연속형은 효과크기가 아예 안 나온다.
**변수 간 순위를 매기려면 모든 변수에 같은 방식으로 적용되는 자가 하나 필요하다.**

In [9]:
def bin7(s):
    "0 / 1 / 2 / 3 / 4-5 / 6-10 / 11+  (설계 §3-3 · 07번과 동일)"
    return cut(s, bins=[-0.5, 0.5, 1.5, 2.5, 3.5, 5.5, 10.5, inf],
               labels=["0", "1", "2", "3", "4-5", "6-10", "11+"], ordered=True)


def bin5(s):
    "0 / 1 / 2 / 3 / 4+  (설계 §3-3 · 07번과 동일)"
    return cut(s, bins=[-0.5, 0.5, 1.5, 2.5, 3.5, inf],
               labels=["0", "1", "2", "3", "4+"], ordered=True)


def binned_of(col):
    "설계 §3-3 이 고정한 구간을 그대로 돌려준다."
    if col in ("prior_view_cnt", "prior_uniq_product_cnt"):
        return bin7(m[col])
    if col == "cart_product_view_cnt":
        return bin5(m[col])
    if col == "price":
        return qcut(m[col], 5, duplicates="drop")      # 5분위
    return m[col]                                      # 이진·명목형은 범주 그대로


def rate_table(col):
    "구간·범주별 Y 비율표. n >= 1,000 여부를 함께 표시한다."
    g = m.groupby(binned_of(col), observed=True)["purchase_7d"]
    t = DataFrame({"고객 수": g.size(), "Y 비율(%)": (g.mean() * 100).round(2)})
    t["n>=1000"] = t["고객 수"] >= MIN_N
    t.index.name = col
    return t


def spread(col):
    "공통 눈금 = n >= 1,000 구간만 쓴 Y 비율 최대 − 최소 (%p)"
    t = rate_table(col)
    v = t.loc[t["n>=1000"], "Y 비율(%)"]
    return round(float(v.max() - v.min()), 2) if len(v) >= 2 else float("nan")


print("구간 함수 준비 완료 — bin7 · bin5 · binned_of · rate_table · spread")
print(f"전체 평균 Y 비율 : {BASE_RATE:.2f}%   ·   공통 눈금 최소 인원 : {MIN_N:,}명")

구간 함수 준비 완료 — bin7 · bin5 · binned_of · rate_table · spread
전체 평균 Y 비율 : 20.50%   ·   공통 눈금 최소 인원 : 1,000명


### 4-1. 변수별 구간·범주 Y 비율표

`n>=1000`이 `False`인 구간은 **공통 눈금 계산에서 빠진다.** 표에는 남겨 두어 왜 빠졌는지 보이게 한다.

In [10]:
for c in X8:
    print("=" * 54)
    print(f"{c}   (전체 평균 {BASE_RATE:.2f}%)")
    display(rate_table(c))

price   (전체 평균 20.50%)


,고객 수,Y 비율(%),n>=1000
price,,,
"(0.049, 2.22]",22009,16.48,True
"(2.22, 4.11]",21428,17.65,True
"(4.11, 6.19]",22275,19.63,True
"(6.19, 11.1]",21116,22.27,True
"(11.1, 327.78]",21630,26.58,True


prior_view_cnt   (전체 평균 20.50%)


,고객 수,Y 비율(%),n>=1000
prior_view_cnt,,,
0,48457,18.54,True
1,33259,21.67,True
2,11781,22.66,True
3,5520,21.85,True
4-5,4841,22.31,True
6-10,3367,22.99,True
11+,1233,25.71,True


prior_uniq_product_cnt   (전체 평균 20.50%)


...

,고객 수,Y 비율(%),n>=1000
brand_grp,,,
bpw.style,2067,12.09,True
cnd,1956,24.85,True
concept,2267,24.48,True
estel,3144,23.89,True
grattol,4741,19.91,True
ingarden,1843,17.09,True
irisk,5457,16.38,True
italwax,2713,28.46,True
jessnail,1535,22.35,True


cart_timeband   (전체 평균 20.50%)


,고객 수,Y 비율(%),n>=1000
cart_timeband,,,
심야,9107,17.78,True
오전,23663,20.73,True
오후,39186,21.80,True
저녁,36502,19.64,True


cart_weekday   (전체 평균 20.50%)


,고객 수,Y 비율(%),n>=1000
cart_weekday,,,
월,15558,20.79,True
화,14730,20.91,True
수,14758,21.52,True
목,15955,21.14,True
금,18235,20.72,True
토,15244,18.98,True
일,13978,19.31,True


### 4-2. 공통 눈금 한 줄 세우기

> ⚠️ **구간 수가 다르면 유리·불리가 갈린다.** `brand_grp`(15종)와 `cart_timeband`(4구간)를
> 이 숫자 하나로 우열을 매기지 않는다. **성격이 비슷한 변수끼리만 비교한다**(설계 §3-3).

In [11]:
SPREAD = DataFrame(
    [{"변수": c,
      "척도": "연속" if c in NUM4 else ("이진" if c == "has_prior_view" else "명목"),
      "구간·범주 수": int(rate_table(c).shape[0]),
      "n>=1000 구간 수": int(rate_table(c)["n>=1000"].sum()),
      "Y비율 최대-최소(%p)": spread(c)} for c in X8]
).set_index("변수").sort_values("Y비율 최대-최소(%p)", ascending=False)
SPREAD

,척도,구간·범주 수,n>=1000 구간 수,Y비율 최대-최소(%p)
변수,,,,
brand_grp,명목,15,15,16.37
cart_product_view_cnt,연속,5,3,10.44
price,연속,5,5,10.10
prior_view_cnt,연속,7,7,7.17
cart_timeband,명목,4,4,4.02
prior_uniq_product_cnt,연속,7,6,3.76
has_prior_view,이진,2,2,3.55
cart_weekday,명목,7,7,2.54


---

## #05. 단조성 — 로지스틱의 전제를 확인한다 (설계 §3-5)

로지스틱 회귀는 **로짓에 직선을 그리는 모형**이다. 관계가 단조가 아니면 계수가 왜곡된다.
이 프로젝트는 이미 U자를 한 번 만났다(번인 전 `prior_view_cnt` 0건 18.87% → 1건 10.13% → 2건 22.43%).

```
단조   구간 순서를 따라 Y 비율이 일관되게 증가 또는 감소한다.
       중간 역전이 있어도 그 폭이 1.0%p 이하이면 노이즈로 보아 단조로 간주한다.
비단조 그 외.  →  ⑤에서 구간화하여 명목형으로 투입한다.
```

> **설계 §3-5에는 `n >= 1,000` 필터가 없다.** 전 구간으로 판정하는 것이 문서대로다.
> 다만 인원이 적은 구간은 흔들리므로, `n >= 1,000`만으로도 함께 재어 **둘이 갈리는지 표시한다.**
> 갈리면 그 사실을 판정표에 남긴다. 어느 쪽을 쓸지 임의로 정하지 않는다.

In [12]:
def monotonic(rates):
    "연속한 구간 사이의 증감을 보고 단조/비단조를 판정한다."
    d = [rates[i + 1] - rates[i] for i in range(len(rates) - 1)]
    if len(d) == 0:
        return "판정불가"
    if all(x >= -NOISE for x in d):
        return "단조↑"
    if all(x <= NOISE for x in d):
        return "단조↓"
    return "비단조"


rows = []
for c in NUM4:                       # 연속형 4종 (설계 §3-5)
    t = rate_table(c)
    full = monotonic(list(t["Y 비율(%)"]))
    big = monotonic(list(t.loc[t["n>=1000"], "Y 비율(%)"]))
    rows.append({"변수": c, "구간 수": len(t),
                 "단조성(전 구간)": full, "단조성(n>=1000만)": big,
                 "두 판정 일치": full == big,
                 "Y비율 나열": " → ".join(f"{v:.2f}" for v in t["Y 비율(%)"])})

MONO = DataFrame(rows).set_index("변수")
MONO

,구간 수,단조성(전 구간),단조성(n>=1000만),두 판정 일치,Y비율 나열
변수,,,,,
price,5,단조↑,단조↑,True,16.48 → 17.65 → 19.63 → 22.27 → 26.58
prior_view_cnt,7,단조↑,단조↑,True,18.54 → 21.67 → 22.66 → 21.85 → 22.31 → 22.99 → 25.71
prior_uniq_product_cnt,7,비단조,비단조,True,18.54 → 22.13 → 22.10 → 20.89 → 22.30 → 21.75 → 26.86
cart_product_view_cnt,5,단조↑,단조↑,True,17.74 → 24.21 → 28.18 → 30.34 → 34.73


---

## #06. 포함관계 규칙 — `prior_view_cnt` vs `has_prior_view` (설계 §3-4)

두 변수는 **포함 관계**다. `has_prior_view`는 `prior_view_cnt`를 0 / 1이상으로 뭉갠 것이므로,
공통 눈금으로 재면 **구간이 많은 쪽이 항상 이긴다.** 비교가 아니라 동어반복이 된다.
그래서 설계 §3-4가 별도의 **상대 규칙**을 미리 정해 두었다.

```
A = prior_view_cnt 의 '1 이상' 구간들끼리의 최대 − 최소
B = '0' vs '1 이상' 차이            ( = has_prior_view 의 두 범주 차이)

A >  B   →  prior_view_cnt 채택   (건수 정보가 필요하다)
A <= B   →  has_prior_view 채택   (있고 없음으로 충분하다)
```

임의의 상수가 없는 상대 규칙이다. 뜻은 —
*"조회를 한 번 했든 열 번 했든 구매율이 비슷하다면, 굳이 건수를 셀 이유가 없다."*

In [13]:
t_cnt = rate_table("prior_view_cnt")
pos = t_cnt.loc[(t_cnt.index != "0") & t_cnt["n>=1000"], "Y 비율(%)"]
A_ = round(float(pos.max() - pos.min()), 2)

t_has = rate_table("has_prior_view")
B_ = round(float(t_has["Y 비율(%)"].max() - t_has["Y 비율(%)"].min()), 2)

keep34 = "prior_view_cnt" if A_ > B_ else "has_prior_view"
drop34 = "has_prior_view" if keep34 == "prior_view_cnt" else "prior_view_cnt"
MARGIN = round(abs(A_ - B_), 2)

print(f"A  '1 이상' 구간들끼리의 최대−최소 : {A_:.2f}%p   "
      f"({', '.join(f'{i}={v:.2f}' for i, v in pos.items())})")
print(f"B  '0' vs '1 이상' 차이            : {B_:.2f}%p   "
      f"({', '.join(f'{i}={v:.2f}' for i, v in t_has['Y 비율(%)'].items())})")
print()
print(f"A {'>' if A_ > B_ else '<='} B   →   설계 §3-4 규칙에 따라  ▶ {keep34} ◀  쪽이 정보를 더 담는다")
print(f"A 와 B 의 차이 : {MARGIN:.2f}%p")
if MARGIN < 1.0:
    print()
    print("⚠️ 차이가 1%p 미만이다. 규칙은 한쪽을 골랐지만 **아슬아슬하게 갈렸다.**")
    print("   이 사실을 #08-2 와 결과 보고서에 반드시 적는다. 표본이 조금만 달랐어도 반대로 갈 수 있었다.")

RULE34 = DataFrame([{"A ('1 이상' 구간 간 최대-최소)": A_,
                     "B ('0' vs '1 이상' 차이)": B_,
                     "차이(%p)": MARGIN, "규칙 결과": keep34}])
RULE34

A  '1 이상' 구간들끼리의 최대−최소 : 4.04%p   (1=21.67, 2=22.66, 3=21.85, 4-5=22.31, 6-10=22.99, 11+=25.71)
B  '0' vs '1 이상' 차이            : 3.55%p   (0=18.54, 1=22.09)

A > B   →   설계 §3-4 규칙에 따라  ▶ prior_view_cnt ◀  쪽이 정보를 더 담는다
A 와 B 의 차이 : 0.49%p

⚠️ 차이가 1%p 미만이다. 규칙은 한쪽을 골랐지만 **아슬아슬하게 갈렸다.**
   이 사실을 #08-2 와 결과 보고서에 반드시 적는다. 표본이 조금만 달랐어도 반대로 갈 수 있었다.


,A ('1 이상' 구간 간 최대-최소),B ('0' vs '1 이상' 차이),차이(%p),규칙 결과
0,4.04,3.55,0.49,prior_view_cnt


---

## #07. 스코어카드 조립 — 설계 §06 형식 그대로

```
| 변수 | 척도 | 단변량 요약 | 검정 | p | 효과크기 | Y비율 최대−최소(%p) | 단조성 | 판정 |
```

`판정` 칸은 `#08`에서 규칙을 적용해 채운다. 여기서는 **근거를 한 줄에 모으기만** 한다.

In [14]:
MULTI = DataFrame([{
    "변수": c,
    "척도": SPREAD.loc[c, "척도"],
    "VIF": round(float(VIF.loc[c, "VIF"]), 3) if c in VIF.index else None,
    "Y비율 최대-최소(%p)": SPREAD.loc[c, "Y비율 최대-최소(%p)"],
    "구간·범주 수": int(SPREAD.loc[c, "구간·범주 수"]),
    "n>=1000 구간 수": int(SPREAD.loc[c, "n>=1000 구간 수"]),
    "단조성": MONO.loc[c, "단조성(전 구간)"] if c in MONO.index else "—",
} for c in X8]).set_index("변수")


def uni_summary(c):
    "연속형 = 평균·중앙값·왜도·0의 비율 / 명목형 = 종수·최대 범주 비율 (설계 §06)"
    s = m[c]
    if c in NUM4:
        return (f"평균 {s.mean():.3f} · 중앙 {s.median():.1f} · "
                f"왜도 {s.skew():.2f} · 0비율 {(s == 0).mean() * 100:.1f}%")
    if c == "has_prior_view":
        return f"2범주 · 1의 비율 {s.mean() * 100:.1f}%"
    return f"{s.nunique()}종 · 최대범주 {s.value_counts(normalize=True).iloc[0] * 100:.1f}%"


CARD = DataFrame([{
    "변수": c,
    "척도": MULTI.loc[c, "척도"],
    "단변량 요약": uni_summary(c),
    "검정": BIV.loc[c, "검정방법"],
    "p": BIV.loc[c, "p"],
    "효과크기": BIV.loc[c, "효과크기"],
    "Y비율 최대-최소(%p)": MULTI.loc[c, "Y비율 최대-최소(%p)"],
    "VIF(8종 전체)": MULTI.loc[c, "VIF"],
    "단조성": MULTI.loc[c, "단조성"],
} for c in X8]).set_index("변수")

CARD.sort_values("Y비율 최대-최소(%p)", ascending=False)

,척도,단변량 요약,검정,p,효과크기,Y비율 최대-최소(%p),VIF(8종 전체),단조성
변수,,,,,,,,
brand_grp,명목,15종 · 최대범주 38.9%,Chi-square test of independence,< 0.0001,V = 0.0608,16.37,NaN,—
cart_product_view_cnt,연속,평균 0.458 · 중앙 0.0 · 왜도 1.75 · 0비율 61.3%,Mann-Whitney U test,< 0.0001,np2 = 0.00776,10.44,2.031,단조↑
price,연속,평균 10.140 · 중앙 5.1 · 왜도 7.21 · 0비율 0.0%,Mann-Whitney U test,< 0.0001,np2 = 0.00080,10.10,1.011,단조↑
prior_view_cnt,연속,평균 1.280 · 중앙 1.0 · 왜도 5.90 · 0비율 44.7%,Mann-Whitney U test,< 0.0001,np2 = 0.00100,7.17,20.218,단조↑
cart_timeband,명목,4종 · 최대범주 36.1%,Chi-square test of independence,< 0.0001,V = 0.0303,4.02,NaN,—
prior_uniq_product_cnt,연속,평균 1.120 · 중앙 1.0 · 왜도 6.04 · 0비율 44.7%,Mann-Whitney U test,< 0.0001,np2 = 0.00073,3.76,19.995,비단조
has_prior_view,이진,2범주 · 1의 비율 55.3%,Chi-square test of independence,< 0.0001,V = 0.0437,3.55,2.051,—
cart_weekday,명목,7종 · 최대범주 16.8%,Chi-square test of independence,< 0.0001,V = 0.0215,2.54,NaN,—


---

## #08. 규칙 적용 — 기계적으로 돌린다

아래 단계는 **전부 결과를 보기 전에 확정된 규칙**이다. 코드가 그대로 실행하고 근거를 출력한다.
**사람이 하는 일은 이 출력이 규칙과 맞는지 확인하는 것**이지, 여기서 새로 판단하는 것이 아니다.

```
0단계  설계 §3-8 시작 배치를 옮긴다  (채택 4 · 보류 4)
1단계  VIF > 10 인 변수를 찾는다                          (설계 §3-9)
2단계  VIF 초과가 낀 쌍 중 가장 강한 쌍에서
       공통 눈금이 작은 쪽을 제외                          (LAB-13 3-7 · 설계 §3-3)
3단계  포함관계 규칙을 적용한다                            (설계 §3-4 · #06)
3-5    제외 후 VIF 를 다시 계산한다
4단계  남은 '보류'를 해소한다                              (설계 §3-8)
```

> **2단계는 한 번만 돈다.** 반복해서 자동으로 깎지 않는다.
> 한 번 제외한 뒤 **3-5에서 임계 이하로 떨어졌는지 검증**하고, 떨어지지 않았으면 `미해소 ❌`가 찍힌다.
> **그때는 사람이 다시 판단한다** — 자동 반복 제거는 설계 §3-9가 금지한 `reduce_vif`와 같은 일이 되기 때문이다.

In [15]:
# ── 0단계 : 설계 §3-8 의 시작 배치를 그대로 옮긴다 ────────────────────
HOLD = ["prior_view_cnt", "prior_uniq_product_cnt",
        "cart_product_view_cnt", "has_prior_view"]      # 오늘 판단할 것

verdict = {c: ("보류" if c in HOLD else "채택") for c in X8}
reason = {c: ("이변량만으로 결정할 수 없다 → 다변량 후 재판정 (설계 §3-8)" if c in HOLD
              else "설계 §3-8 시작 배치에서 채택 확정 · 이변량 결과로 뒤집지 않는다") for c in X8}

print("【0단계】 설계 §3-8 시작 배치")
print(f"   채택 : {[c for c in X8 if verdict[c] == '채택']}")
print(f"   보류 : {[c for c in X8 if verdict[c] == '보류']}")

【0단계】 설계 §3-8 시작 배치
   채택 : ['price', 'brand_grp', 'cart_timeband', 'cart_weekday']
   보류 : ['prior_view_cnt', 'prior_uniq_product_cnt', 'cart_product_view_cnt', 'has_prior_view']


In [16]:
# ── 1단계 : VIF 초과 ────────────────────────────────────────────────
over = MULTI[(MULTI["VIF"].notna()) & (MULTI["VIF"] > VIF_MAX)]
print("【1단계】 VIF > 10.0 인 변수")
for c, r in over.iterrows():
    print(f"   {c:<24} VIF {r['VIF']:.3f}   (#03)")
if len(over) == 0:
    print("   없음 — 중복으로 자를 근거가 없다")

【1단계】 VIF > 10.0 인 변수
   prior_view_cnt           VIF 20.218   (#03)
   prior_uniq_product_cnt   VIF 19.995   (#03)


In [17]:
# ── 2단계 : VIF 초과가 낀 쌍 중 가장 강한 쌍에서 눈금이 작은 쪽을 제외 ──
CORRP = CORR.reset_index()
gauge = MULTI["Y비율 최대-최소(%p)"].to_dict()

print("【2단계】 중복 해소 — 효과 강도가 더 높은 쪽을 남긴다 (LAB-13 3-7)")
if len(over):
    # VIF 초과 변수가 낀 쌍만 후보다. 초과와 무관한 쌍을 건드리면 규칙을 벗어난다.
    cand = CORRP[CORRP["x"].isin(over.index) | CORRP["y"].isin(over.index)]
    pair = cand.loc[cand["coef"].abs().idxmax()]
    x, y, coef = pair["x"], pair["y"], float(pair["coef"])
    loser = x if gauge[x] <= gauge[y] else y
    winner = y if loser == x else x
    print(f"   가장 강한 쌍 : {x} ↔ {y}   coef {coef:.4f}  ({pair['method']}, #02)")
    print(f"   공통 눈금    : {x} {gauge[x]:.2f}%p   vs   {y} {gauge[y]:.2f}%p  (#04-2)")
    print(f"   단조성       : {x} {MULTI.loc[x, '단조성']}   vs   {y} {MULTI.loc[y, '단조성']}  (#05)")
    print(f"   → 눈금이 작은 {loser} 를 제외하고 {winner} 를 남긴다")
    verdict[loser] = "제외"
    reason[loser] = (f"다변량 중복 — {winner} 와 coef {coef:.4f}, VIF {MULTI.loc[loser, 'VIF']:.2f}. "
                     f"공통 눈금 {gauge[loser]:.2f}%p < {gauge[winner]:.2f}%p 라 강도가 낮은 쪽 "
                     f"(설계 §3-8 근거② · §3-9)")
else:
    print("   해당 없음")

【2단계】 중복 해소 — 효과 강도가 더 높은 쪽을 남긴다 (LAB-13 3-7)
   가장 강한 쌍 : prior_view_cnt ↔ prior_uniq_product_cnt   coef 0.9881  (Spearman, #02)
   공통 눈금    : prior_view_cnt 7.17%p   vs   prior_uniq_product_cnt 3.76%p  (#04-2)
   단조성       : prior_view_cnt 단조↑   vs   prior_uniq_product_cnt 비단조  (#05)
   → 눈금이 작은 prior_uniq_product_cnt 를 제외하고 prior_view_cnt 를 남긴다


In [18]:
# ── 3단계 : 포함관계 규칙 (설계 §3-4) — #06 결과를 그대로 적용 ────────
print("【3단계】 포함관계 — prior_view_cnt ↔ has_prior_view (설계 §3-4)")
print(f"   A {A_:.2f}%p  {'>' if A_ > B_ else '<='}  B {B_:.2f}%p   (차이 {MARGIN:.2f}%p, #06)")
print(f"   → {keep34} 채택 · {drop34} 제외")
if MARGIN < 1.0:
    print("   ⚠️ 차이가 1%p 미만 — 아슬아슬하게 갈렸다. #08-2 와 보고서에 적는다")

if verdict.get(drop34) != "제외":
    verdict[drop34] = "제외"
    reason[drop34] = (f"포함관계 중복 — {keep34} 를 0/1 로 뭉갠 변수. "
                      f"설계 §3-4 규칙 A {A_:.2f} {'>' if A_ > B_ else '<='} B {B_:.2f} "
                      f"(차이 {MARGIN:.2f}%p) → {keep34} 채택 (설계 §3-8 근거②)")

【3단계】 포함관계 — prior_view_cnt ↔ has_prior_view (설계 §3-4)
   A 4.04%p  >  B 3.55%p   (차이 0.49%p, #06)
   → prior_view_cnt 채택 · has_prior_view 제외
   ⚠️ 차이가 1%p 미만 — 아슬아슬하게 갈렸다. #08-2 와 보고서에 적는다


In [19]:
# ── 3-5 : 제외 후 VIF 재계산 ────────────────────────────────────────
# VIF 는 X 집합 전체에서 나오는 값이라, 변수를 빼면 나머지가 전부 다시 계산된다.
# 그러므로 '보류를 채택으로 올려도 되는가'는 반드시 '제외 후' VIF 로 판단해야 한다.
kept_num = [c for c in NUM5 if verdict[c] != "제외"]
VIF2 = my_stats.compute_vif(m, columns=kept_num)
vif_after = VIF2["VIF"].to_dict()

print("【3-5】 제외 후 VIF 재계산")
print(f"   남은 수치형 : {kept_num}")
for c in kept_num:
    print(f"   {c:<24} {MULTI.loc[c, 'VIF']:>7.3f}  →  {vif_after[c]:>6.3f}")
print(f"   최대 VIF {VIF2['VIF'].max():.3f}  (임계 {VIF_MAX})  → "
      f"{'해소 ✅' if VIF2['VIF'].max() <= VIF_MAX else '미해소 ❌ 재검토 필요'}")
VIF2.round(3)

【3-5】 제외 후 VIF 재계산
   남은 수치형 : ['price', 'prior_view_cnt', 'cart_product_view_cnt']
   price                      1.011  →   1.011
   prior_view_cnt            20.218  →   1.237
   cart_product_view_cnt      2.031  →   1.249
   최대 VIF 1.249  (임계 10.0)  → 해소 ✅


,VIF
cart_product_view_cnt,1.249
prior_view_cnt,1.237
price,1.011


In [20]:
# ── 4단계 : 남은 '보류'를 해소한다 ──────────────────────────────────
# 판단 근거는 '제외 후' VIF 다. 제외 전 값(#03)을 쓰면 이미 해소된 중복을 근거로 삼게 된다.
print("【4단계】 보류 해소  — 근거는 제외 후 VIF (#08 3-5)")
still = [c for c in X8 if verdict[c] == "보류"]
for c in still:
    v = vif_after.get(c)
    okv = (v is not None) and (v <= VIF_MAX)
    verdict[c] = "채택" if okv else "보류"
    if okv:
        reason[c] = (f"보류 → 채택. 제외 후 VIF {v:.2f} 로 임계 이하 "
                     f"(제외 전 {MULTI.loc[c, 'VIF']:.2f}). 중복이 해소돼 자를 근거가 없다 "
                     f"(설계 §3-8 — 자르는 근거는 중복뿐)")
        print(f"   {c:<24} 보류 → 채택   (제외 후 VIF {v:.3f} <= {VIF_MAX})")
    else:
        print(f"   {c:<24} 보류 유지 ❌  (제외 후 VIF {v})  → 사람이 사유를 적고 결정한다")
if not still:
    print("   남은 보류 없음")

【4단계】 보류 해소  — 근거는 제외 후 VIF (#08 3-5)
   prior_view_cnt           보류 → 채택   (제외 후 VIF 1.237 <= 10.0)
   cart_product_view_cnt    보류 → 채택   (제외 후 VIF 1.249 <= 10.0)


### 8-1. 판정을 붙인 스코어카드 — 설계 §06 완성형

In [21]:
CARD["VIF(제외 후)"] = [round(vif_after[c], 3) if c in vif_after else None for c in CARD.index]
CARD["판정"] = [verdict[c] for c in CARD.index]
CARD["판정 근거"] = [reason[c] for c in CARD.index]

order = {"채택": 0, "보류": 1, "제외": 2}
SCORECARD = CARD.sort_values(
    ["판정", "Y비율 최대-최소(%p)"],
    key=lambda s: s.map(order) if s.name == "판정" else -s,
)
SCORECARD[["척도", "검정", "p", "효과크기", "Y비율 최대-최소(%p)",
           "VIF(8종 전체)", "VIF(제외 후)", "단조성", "판정"]]

,척도,검정,p,효과크기,Y비율 최대-최소(%p),VIF(8종 전체),VIF(제외 후),단조성,판정
변수,,,,,,,,,
brand_grp,명목,Chi-square test of independence,< 0.0001,V = 0.0608,16.37,NaN,NaN,—,채택
cart_product_view_cnt,연속,Mann-Whitney U test,< 0.0001,np2 = 0.00776,10.44,2.031,1.249,단조↑,채택
price,연속,Mann-Whitney U test,< 0.0001,np2 = 0.00080,10.10,1.011,1.011,단조↑,채택
prior_view_cnt,연속,Mann-Whitney U test,< 0.0001,np2 = 0.00100,7.17,20.218,1.237,단조↑,채택
cart_timeband,명목,Chi-square test of independence,< 0.0001,V = 0.0303,4.02,NaN,NaN,—,채택
cart_weekday,명목,Chi-square test of independence,< 0.0001,V = 0.0215,2.54,NaN,NaN,—,채택
prior_uniq_product_cnt,연속,Mann-Whitney U test,< 0.0001,np2 = 0.00073,3.76,19.995,NaN,비단조,제외
has_prior_view,이진,Chi-square test of independence,< 0.0001,V = 0.0437,3.55,2.051,NaN,—,제외


In [22]:
print("판정 요약")
for k in ["채택", "보류", "제외"]:
    v = [c for c in X8 if verdict[c] == k]
    print(f"  {k} {len(v)}종 : {', '.join(v) if v else '—'}")
print()
print("※ '보류'가 남아 있으면 규칙이 결정을 못 낸 것이다. 그때는 사람이 사유를 적고 결정한다.")

판정 요약
  채택 6종 : price, prior_view_cnt, cart_product_view_cnt, brand_grp, cart_timeband, cart_weekday
  보류 0종 : —
  제외 2종 : prior_uniq_product_cnt, has_prior_view

※ '보류'가 남아 있으면 규칙이 결정을 못 낸 것이다. 그때는 사람이 사유를 적고 결정한다.


### 8-2. 계단 구조 점검 — 남긴 변수를 그대로 넣어도 되는가

조회 변수들은 단조로 판정됐지만, **0 → 1 구간의 계단이 그 뒤 전 구간의 변화폭과 맞먹는지** 확인한다.
계단이 크면 **단조이긴 해도 직선이 아니다.** 로지스틱은 로짓에 직선을 긋는 모형이라
이 경우 계수가 평균적인 기울기만 잡고 계단을 놓친다.

> **0에 덩어리가 있는 카운트 변수에만 뜻이 있다.** `price`는 ③에서 0을 이미 제거해
> *"0 vs 1 이상"*이라는 계단 자체가 존재하지 않으므로 대상에서 뺀다.
>
> ⚠️ 이 점검은 **설계 문서에 없는 추가 관찰**이다. **제외 사유가 아니고 판정을 바꾸지도 않는다.**
> ⑤에서 구간화 투입을 검토할지 판단하는 재료로만 쓴다.

In [23]:
STEP_TARGET = [x for x in X8
               if verdict[x] == "채택" and x in NUM4 and (m[x] == 0).mean() > 0]

step_rows = []
for c in STEP_TARGET:
    r = m.groupby(m[c].clip(upper=1), observed=True)["purchase_7d"].mean() * 100
    step = round(float(r.iloc[1] - r.iloc[0]), 2)
    total = float(MULTI.loc[c, "Y비율 최대-최소(%p)"])
    ratio = step / total if total else float("nan")
    step_rows.append({"변수": c,
                      "0의 비율(%)": round((m[c] == 0).mean() * 100, 1),
                      "0 → 1이상 계단(%p)": step,
                      "전체 폭(%p)": total,
                      "계단 비중(%)": round(ratio * 100, 1),
                      "계단이 절반 이상": bool(ratio >= 0.5)})
STEP = DataFrame(step_rows).set_index("변수")

skipped = [x for x in X8 if verdict[x] == "채택" and x in NUM4 and x not in STEP_TARGET]
print("계단 비중이 50%를 넘으면 '단조이지만 직선은 아니다'. ⑤에서 구간화를 검토한다.")
if skipped:
    print(f"대상 제외 : {skipped} — 0 값이 없어 '0 vs 1이상' 계단이 성립하지 않는다")
STEP

계단 비중이 50%를 넘으면 '단조이지만 직선은 아니다'. ⑤에서 구간화를 검토한다.
대상 제외 : ['price'] — 0 값이 없어 '0 vs 1이상' 계단이 성립하지 않는다


,0의 비율(%),0 → 1이상 계단(%p),전체 폭(%p),계단 비중(%),계단이 절반 이상
변수,,,,,
prior_view_cnt,44.7,3.55,7.17,49.5,False
cart_product_view_cnt,61.3,7.14,10.44,68.4,True


---

## #09. 확인과 서술 — **둘이 함께 읽고 채운다**

`#08`은 **미리 정한 규칙을 코드가 그대로 돌린 결과**다. 사람이 할 일은 새로 판단하는 게 아니라
**규칙이 의도대로 적용됐는지 확인**하고, 그 결과를 문장으로 남기는 것이다.

### 9-1. 규칙 적용 확인

| # | 확인할 것 | 어디서 | 확인 |
|---|---|---|---|
| 1 | VIF 초과 변수가 `#08` 1단계와 `#03`에서 같은가 | `#08` · `#03` | ✅ 양쪽 모두 `prior_view_cnt`(20.218) · `prior_uniq_product_cnt`(19.995) 두 개뿐 |
| 2 | 2단계가 **공통 눈금**으로 골랐는가 (VIF 크기로 고르지 않았는가) | `#08` | ✅ 눈금 `prior_view_cnt` 7.17%p vs `prior_uniq_product_cnt` 3.76%p 로 갈랐다. VIF 는 20.22·20.00 으로 사실상 같아 기준이 못 된다 |
| 3 | 3단계 A·B 값이 `#06` 출력과 같은가 | `#08` · `#06` | ✅ A 4.04%p · B 3.55%p · 차이 0.49%p 로 일치 |
| 4 | 제외 후 최대 VIF가 10 이하로 떨어졌는가 | `#08` 3-5 | ✅ 20.218 → 1.249 (`해소 ✅`) |
| 5 | 상관(Spearman)과 VIF(선형)가 같은 방향을 가리키는가 | `#02` · `#03` | ✅ Spearman 0.9881(`prior_view_cnt`↔`prior_uniq_product_cnt`) · 0.9177(↔`has_prior_view`) = VIF 초과와 같은 방향. `price` 는 상관 Weak(≤0.12) = VIF 1.011 |

---

### 9-2. 다변량 관찰

| 항목 | 관찰 내용 |
|---|---|
| 계수 종류 | **10쌍 전부 Spearman.** 5종 모두 `high_skew=True`(왜도 1.75~7.21)라 자동 전환됐다 (`#02` `method` 열) |
| 중간 이상(\|coef\| > 0.3) 쌍 | **6** 개 · 가장 강한 쌍 `prior_view_cnt` ↔ `prior_uniq_product_cnt` coef **0.9881** (`#02-1`) |
| 최대 VIF · 임계 초과 변수 | 최대 **20.218** · 초과 = `prior_view_cnt`(20.218) · `prior_uniq_product_cnt`(19.995) 2종 (`#03`) |
| 하나를 뺐을 때 | `prior_uniq_product_cnt` 를 빼면 최대 VIF 가 **1.817** 로 떨어진다 (`prior_view_cnt` 를 빼도 1.922) (`#03-1`) |
| 공통 눈금 최대 / 최소 | `brand_grp` **16.37**%p / `cart_weekday` **2.54**%p (`#04-2`) |
| 비단조 변수 | `prior_uniq_product_cnt` 하나 (전 구간·`n≥1000`만 둘 다 비단조). 나머지 연속형 3종은 단조↑ (`#05`) |
| 포함관계 규칙 결과 | A **4.04**%p vs B **3.55**%p → **`prior_view_cnt` 채택** · 차이 **0.49**%p (`#06`) |

**소견** : 다중공선성은 **조회·장바구니 카운트 4종 안에서만** 나타난다 — `price` 와 명목형 3종은 이들과 무관하다(상관 ≤0.12 · VIF ≤1.01). `prior_uniq_product_cnt` 는 `prior_view_cnt` 와 사실상 같은 변수(Spearman 0.9881)이고, `has_prior_view` 는 `prior_view_cnt` 를 0/1 로 뭉갠 것이다. 이 둘을 제외하면 최대 VIF 가 20.2 → 1.25 로 내려가고 남은 6종은 전부 임계 이하다(`#08` 3-5). Spearman(순위)과 VIF(선형)가 같은 방향을 가리켜 근거가 겹친다. 다만 포함관계 규칙이 0.49%p 차이로 아슬아슬하게 갈린 점은 한계로 남긴다(9-5).

---

### 9-3. 제외한 변수 — 왜 뺐는가

> 계획 보고서 §3에 *"왜 넣는가"*를 써서 제출한 변수를 빼는 것이므로, **뺀 이유를 반드시 남긴다.**

| 변수 | 제외 사유 | 확인 |
|---|---|---|
| `prior_uniq_product_cnt` | 다변량 중복 — `prior_view_cnt` 와 Spearman 0.9881 · VIF 19.995. 공통 눈금 3.76%p < 7.17%p 이고 비단조라 강도가 낮은 쪽을 제외 (`#08` 2단계 · 설계 §3-8 근거② · §3-9) | ✅ |
| `has_prior_view` | 포함관계 중복 — `prior_view_cnt` 를 0/1 로 뭉갠 변수. 설계 §3-4 규칙 A 4.04 > B 3.55 (차이 0.49%p) → `prior_view_cnt` 채택 (`#08` 3단계) | ✅ |

---

### 9-4. 남긴 변수 — 해석이 되는가

결과 보고서 사양 §5 — *"설명이 목적이면 변수 추가 기준은 **해석이 되는가**"*.
최종 X 각각에 대해 **계수를 어떻게 읽을 것인지** 한 줄로 적는다.

| 변수 | 계수를 어떻게 읽을 것인가 |
|---|---|
| `price` | `log` 변환 후 투입. 담은 상품 가격이 높을수록 구매 승산이 오른다 — 5분위로 16.48% → 26.58% 단조 증가. 비싼 상품을 담은 고객일수록 구매 의도가 뚜렷하다 |
| `prior_view_cnt` | `log1p` 변환 후 투입. 담기 전 사전 조회가 많을수록 구매 승산 증가. 단 0→1 계단(3.55%p)이 전체 폭의 49.5% — "조회 이력 유무" 자체의 효과가 크다 |
| `cart_product_view_cnt` | `log1p` 변환. 담은 그 상품을 사전에 본 횟수. 0→1 계단이 전체 폭의 68.4%(`#08-2`)라 로짓 직선 가정이 약함 → ⑤에서 구간화 검토, 계수는 "평균 기울기"로만 읽는다 |
| `brand_grp` | 더미(분할 후). 기준 브랜드 대비 각 브랜드의 구매 승산 차. 효과는 작지만(V=0.0608) `price` 와의 교란 통제용이라 유지한다 |
| `cart_timeband` | 더미(분할 후). 기준 시간대 대비 심야·오전·저녁의 구매 승산. 심야가 가장 낮다(17.78% vs 오후 21.80%) |
| `cart_weekday` | 더미(분할 후). 요일 효과는 매우 작다(V=0.0215 · 폭 2.54%p). 토·일이 소폭 낮은 정도로만 읽는다 |

---

### 9-5. 이 판정의 한계

| 한계 | 내용 |
|---|---|
| 공통 눈금은 구간 수에 좌우된다 | `brand_grp`(15종)가 `cart_timeband`(4구간)보다 구조적으로 유리하다. **절대 크기 판정에 쓰지 않았다**(설계 §3-3) |
| 명목형 VIF 를 아직 못 봤다 | 더미 전개 후 ⑤에서 확인한다. **여기서 '중복 없음'이라 말할 수 있는 것은 수치형뿐**이다(설계 §3-9) |
| 상관과 VIF 는 다른 자다 | 상관은 Spearman(순위), VIF는 선형회귀 기반이다. 이번에 같은 방향이었더라도 **근거가 겹친 것이지 같은 것을 잰 게 아니다** |
| 포함관계 규칙이 아슬아슬하게 갈렸다 | `#06`의 `A − B` 차이가 **0.49%p** 로 1%p 미만이다. 표본이 조금만 달랐어도 `has_prior_view` 쪽으로 갈 수 있었다 |
| `cart_product_view_cnt` 는 단조지만 직선이 아니다 | 채택했으나 0→1 계단이 전체 폭의 68.4%(`#08-2`). 로지스틱 계수는 계단을 놓친다 → ⑤ 구간화 검토 대상으로 넘긴다 |


---

## #10. 최종 X 집합과 ⑤로 넘기는 지시

### 10-1. 로그변환 판정 (설계 §3-6) — **판정만 한다. 실행은 ⑤**

```
대상 : |왜도| >= 1.0        방법 : 0을 포함하면 log1p, 아니면 log
```

> **왜도는 반드시 마스터 테이블(108,458행) 기준으로 잰다.** 원본 로그(825만 행)의 `price` 왜도는
> *"각 고객이 첫 담기 때 본 가격"*과 **다른 분포의 값**이다(설계 §3-6). `#01-3`에서 09번과 대조했다.

In [24]:
FINAL_X = [c for c in X8 if verdict[c] == "채택"]

rows = []
for c in [x for x in FINAL_X if x in NUM4]:
    sk = float(m[c].skew())
    has0 = bool((m[c] == 0).any())
    rows.append({"변수": c, "왜도": round(sk, 2),
                 "대상(|왜도|>=1.0)": abs(sk) >= SKEW_MAX,
                 "방법": ("log1p" if has0 else "log") if abs(sk) >= SKEW_MAX else "—",
                 "0 포함": has0,
                 "단조성": MULTI.loc[c, "단조성"]})
LOGPLAN = DataFrame(rows).set_index("변수")
LOGPLAN

,왜도,대상(|왜도|>=1.0),방법,0 포함,단조성
변수,,,,,
price,7.21,True,log,False,단조↑
prior_view_cnt,5.90,True,log1p,True,단조↑
cart_product_view_cnt,1.75,True,log1p,True,단조↑


### 10-2. ⑤ 전처리에 넘기는 지시서

순서를 지켜야 정보 누수가 없다 — **층화 분할 → 변환 → 인코딩 → 스케일링** (LAB-13 04).

In [25]:
todo = []
for c in FINAL_X:
    t = []
    if c in LOGPLAN.index and LOGPLAN.loc[c, "대상(|왜도|>=1.0)"]:
        t.append(f"{LOGPLAN.loc[c, '방법']} 변환")
    if MULTI.loc[c, "단조성"] == "비단조":
        t.append("구간화 후 명목형 투입 (설계 §3-5)")
    elif c in STEP.index and bool(STEP.loc[c, "계단이 절반 이상"]):
        t.append("구간화 검토 — 단조이나 0→1 계단이 전체 폭의 절반 이상 (#08-2)")
    if MULTI.loc[c, "척도"] == "명목":
        t.append("더미 인코딩 (분할 후)")
    todo.append({"변수": c, "척도": MULTI.loc[c, "척도"],
                 "⑤에서 할 일": " · ".join(t) if t else "그대로 투입"})

TODO5 = DataFrame(todo).set_index("변수")
print(f"최종 X : {len(FINAL_X)}종   {FINAL_X}")
print(f"제외   : {[c for c in X8 if verdict[c] == '제외']}")
print()
print("이상치 : 처리하지 않는다. price 의 큰 값은 오류가 아니라 비싼 상품이고,")
print("         진짜 오류(음수·0원)는 ③에서 이미 제거했다 (설계 §3-7)")
TODO5

최종 X : 6종   ['price', 'prior_view_cnt', 'cart_product_view_cnt', 'brand_grp', 'cart_timeband', 'cart_weekday']
제외   : ['prior_uniq_product_cnt', 'has_prior_view']

이상치 : 처리하지 않는다. price 의 큰 값은 오류가 아니라 비싼 상품이고,
         진짜 오류(음수·0원)는 ③에서 이미 제거했다 (설계 §3-7)


,척도,⑤에서 할 일
변수,,
price,연속,log 변환
prior_view_cnt,연속,log1p 변환
cart_product_view_cnt,연속,log1p 변환 · 구간화 검토 — 단조이나 0→1 계단이 전체 폭의 절반 이상 (#08-2)
brand_grp,명목,더미 인코딩 (분할 후)
cart_timeband,명목,더미 인코딩 (분할 후)
cart_weekday,명목,더미 인코딩 (분할 후)


### 10-3. 저장

| 파일 | 쓰임 |
|---|---|
| `11_스코어카드.csv` | 결과 보고서 §4 에 그대로 들어간다 |
| `11_최종X.csv` | ⑤ 전처리 노트북이 읽어 이 목록대로만 처리한다 |
| `11_다변량.csv` · `11_상관행렬.csv` · `11_포함관계규칙.csv` · `11_로그변환판정.csv` | 근거 원본 |

In [26]:
SCORECARD.to_csv(OUT + r"\11_스코어카드.csv", encoding="utf-8-sig")
TODO5.assign(판정="채택").to_csv(OUT + r"\11_최종X.csv", encoding="utf-8-sig")
MULTI.to_csv(OUT + r"\11_다변량.csv", encoding="utf-8-sig")
CORR.to_csv(OUT + r"\11_상관행렬.csv", encoding="utf-8-sig")
RULE34.to_csv(OUT + r"\11_포함관계규칙.csv", index=False, encoding="utf-8-sig")
LOGPLAN.to_csv(OUT + r"\11_로그변환판정.csv", encoding="utf-8-sig")

print("저장 완료")
print(f"  11_스코어카드.csv     {SCORECARD.shape[0]}행 × {SCORECARD.shape[1]}열")
print(f"  11_최종X.csv          {TODO5.shape[0]}행")
print(f"  11_다변량.csv         {MULTI.shape[0]}행")
print(f"  11_상관행렬.csv       {CORR.shape[0]}쌍")
print(f"  11_포함관계규칙.csv   A={A_} · B={B_} · 결과={keep34}")
print(f"  11_로그변환판정.csv   {LOGPLAN.shape[0]}행")

저장 완료
  11_스코어카드.csv     8행 × 11열
  11_최종X.csv          6행
  11_다변량.csv         8행
  11_상관행렬.csv       10쌍
  11_포함관계규칙.csv   A=4.04 · B=3.55 · 결과=prior_view_cnt
  11_로그변환판정.csv   3행


---

## #11. 다음

| 할 일 | 누가 |
|---|---|
| `#09` 확인 5개 · 관찰 · 제외 사유 · 해석 한 줄 채우기 | **둘이 함께** |
| 결과 보고서 §4 에 스코어카드와 한계 옮기기 | 둘이 함께 |
| ⑤ 모델링용 전처리 — **층화 분할 먼저** | 둘이 함께 |
| 명목형 더미 전개 후 VIF 재확인 | ⑤ |
| ⑥ 로지스틱 회귀 기준선 → 트리 비교 | ⑥ |

### 결과 보고서 §4 에 반드시 옮길 것

| 옮길 것 | 어디서 |
|---|---|
| 스코어카드 표 (설계 §06 형식) | `#08-1` |
| 제외한 변수와 그 사유 | `#09-3` |
| 공통 눈금이 구간 수에 좌우된다는 한계 | `#09-5` (설계 §3-3 이 *"보고서에도 한 줄 적는다"*고 지시) |
| 명목형 VIF 는 ⑤에서 본다는 사실 | `#09-5` |
| 검정이 전부 비모수로 자동 전환된 이유 | `10a` · `10b` |

> **효과크기가 전부 미미 구간이라는 사실도 그대로 적는다.** 숨기면 안 되고, 숨길 이유도 없다 —
> 설계 §3-2가 *"효과크기가 Cohen 해석선을 넘으면 오히려 정보 누수를 의심한다"*고 미리 적어 두었다.

> **④ 탐색적 데이터 분석은 여기서 끝난다.** 다음 노트북부터는 ⑤ 모델링용 전처리다.